# 🤖 Agent Development — Risk Analyst & Compliance Officer

This notebook develops and tests both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 📋 What This Notebook Covers
- Chain-of-Thought prompting for systematic reasoning
- ReACT prompting for structured action-taking
- Structured JSON outputs from LLMs
- Error handling and validation
- Testing agents with real financial data

## 🚀 Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
from dotenv import load_dotenv
from datetime import datetime

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../env.txt')

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path:", os.path.abspath('../src'))

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Source directory added to Python path: /Users/ak/Documents/aiforfin/project/starter/src


In [2]:
# OpenAI client setup
import openai

openai_api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
else:
    client = openai.OpenAI(api_key=openai_api_key, base_url=base_url if base_url else None)
    print("✅ OpenAI client initialized")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    if base_url:
        print(f"📍 Base URL: {base_url}")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: voc-2045...0158
📍 Base URL: https://openai.vocareum.com/v1

💡 Tip: After implementing foundation_sar.py, you can use:
   from src import create_vocareum_openai_client
   client = create_vocareum_openai_client()


## 📊 Load Foundation Components

Ensure the foundation module is working before building agents.

In [3]:
# Import foundation components (implemented in foundation_sar.py)

from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)

print("✅ Foundation components imported")
print("✅ Ready to create sample cases for agent testing")

✅ Foundation components imported
✅ Ready to create sample cases for agent testing


In [4]:
# Test foundation components: load CSV, create DataLoader, generate sample case

print("✅ Loading CSV data from ../data/")
print("✅ ExplainabilityLogger and DataLoader created")
print("✅ Sample case generated for agent testing")
customers_df, accounts_df, transactions_df = load_csv_data("../data/")
os.makedirs("../outputs/audit_logs", exist_ok=True)
logger = ExplainabilityLogger("../outputs/audit_logs/agent_development.jsonl")
loader = DataLoader(logger)
first_customer = customers_df.iloc[1].to_dict()
all_accounts   = accounts_df.to_dict(orient='records')
all_transactions = transactions_df.to_dict(orient='records')

sample_case = loader.create_case_from_data(
    customer_data=first_customer,
    account_data=all_accounts,
    transaction_data=all_transactions
)
print(sample_case)
# Example workflow:
# logger = ExplainabilityLogger("../outputs/audit_logs/agent_development.jsonl")
# loader = DataLoader(logger)
# customers_df, accounts_df, transactions_df = load_csv_data("../data/")
# # Create sample case...

✅ Loading CSV data from ../data/
✅ ExplainabilityLogger and DataLoader created
✅ Sample case generated for agent testing
case_id='e2481ed6-9524-4908-a29a-3db13ab8f8d2' customer=CustomerData(customer_id='CUST_0002', name='Renee Blair', date_of_birth='1978-05-14', ssn_last_4='7201', address='959 Janet Cape Apt. 413, South Joshuastad, GA 49021', customer_since='2023-03-03', risk_rating='Low', phone='(534)719-2832x764', occupation='Banker', annual_income=72005) accounts=[AccountData(account_id='CUST_0002_ACC_1', customer_id='CUST_0002', account_type='Money_Market', opening_date='2023-12-03', current_balance=92121.66, average_monthly_balance=172242.96, status='Active')] transactions=[TransactionData(transaction_id='TXN_B24455F3', account_id='CUST_0002_ACC_1', transaction_date='2025-02-18', transaction_type='Online_Transfer', amount=1615.06, description='Credit card payment', method='ATM', counterparty=None, location='Branch_Westside_Plaza'), TransactionData(transaction_id='TXN_0A18BAD4', ac

## 🔍 Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear steps
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
# Chain-of-Thought prompt design (implemented in risk_analyst_agent.py)
# Design and test the CoT prompt structure

def design_cot_prompt():
    """Design and test Chain-of-Thought prompt for risk analysis"""
    system_prompt = """
    You are a Senior Financial Crime Risk Analyst with 15+ years of experience in AML compliance, 
    fraud detection, and regulatory reporting. You specialize in identifying suspicious activity 
    patterns and filing Suspicious Activity Reports (SARs) in accordance with BSA/AML regulations.

    When analyzing a case, you MUST follow this exact 5-step Chain-of-Thought framework:

    STEP 1 - DATA REVIEW:
    - Examine customer profile (identity, risk rating, occupation, income)
    - Review account details (type, balance, account age, status)
    - Catalog all transactions (amounts, types, methods, counterparties, dates)
    - Note any missing or suspicious data points

    STEP 2 - PATTERN RECOGNITION:
    - Identify unusual transaction volumes or frequencies
    - Look for structuring patterns (transactions just below $10,000)
    - Check for rapid movement of funds between accounts
    - Identify unusual counterparties or geographic locations
    - Compare transaction behavior against customer's income and occupation

    STEP 3 - REGULATORY MAPPING:
    - Map findings to relevant regulations:
      * BSA (31 CFR 1020.320) - SAR filing requirements
      * FinCEN guidelines - suspicious activity indicators
      * OFAC - sanctions screening
      * 12 CFR 21.11 - national bank SAR requirements
    - Identify which regulatory thresholds have been triggered

    STEP 4 - RISK QUANTIFICATION:
    - Assign confidence score between 0.0 and 1.0
    - Consider: volume of suspicious indicators found
    - Consider: customer risk rating (Low/Medium/High)
    - Consider: transaction amounts relative to reported income
    - Assign overall risk level: Low, Medium, High, or Critical

    STEP 5 - CLASSIFICATION DECISION:
    - Select ONE primary classification:
      * Structuring: Transactions deliberately kept below $10,000 to avoid reporting
      * Sanctions: Transactions involving OFAC-listed entities or countries
      * Fraud: Deceptive transactions or identity-related suspicious activity
      * Money_Laundering: Layering or integrating illicit funds
      * Other: Suspicious activity that doesn't fit above categories
    - Summarize key indicators that led to this decision

    OUTPUT FORMAT:
    You must respond with a valid JSON object and nothing else. No preamble, no explanation outside the JSON.
    Use exactly this structure:
    {
        "classification": "Structuring" | "Sanctions" | "Fraud" | "Money_Laundering" | "Other",
        "confidence_score": 0.0 to 1.0,
        "reasoning": "Your step-by-step analysis (max 500 characters)",
        "key_indicators": ["indicator 1", "indicator 2", "indicator 3"],
        "risk_level": "Low" | "Medium" | "High" | "Critical"
    }
    """
    return system_prompt

# Test the prompt design
cot_prompt = design_cot_prompt()
print("🧠 Chain-of-Thought Prompt Design:")
print(cot_prompt[:200] + "...")
print("\n✅ CoT prompt implemented in risk_analyst_agent.py")

🧠 Chain-of-Thought Prompt Design:

    You are a Senior Financial Crime Risk Analyst with 15+ years of experience in AML compliance, 
    fraud detection, and regulatory reporting. You specialize in identifying suspicious activity 
  ...

✅ CoT prompt implemented in risk_analyst_agent.py


In [6]:
from risk_analyst_agent import RiskAnalystAgent

def simple_risk_analyst_smoke_test(client):
    print("🔍 Risk Analyst Smoke Test")
    print("=" * 45)

    # ── Step 1: Initialize agent ──────────────────────────────────────────────
    try:
        logger = ExplainabilityLogger(log_file="smoke_test_audit.jsonl")
        agent  = RiskAnalystAgent(
            openai_client=client,
            explainability_logger=logger,
            model="gpt-4o"
        )
        print("✅ Step 1 — Agent initialized")

    except Exception as e:
        print(f"❌ Step 1 FAILED — Could not initialize agent: {e}")
        return

    # ── Step 2: Build minimal test case ──────────────────────────────────────
    try:
        customer = CustomerData(
            customer_id="CUST_SMOKE_001",
            name="Test User",
            date_of_birth="1975-04-20",
            ssn_last_4="1234",
            address="789 Test St, Newark, NJ 07102",
            customer_since="2020-01-01",
            risk_rating="High",
            occupation="Cashier",
            annual_income=28000
        )

        account = AccountData(
            account_id="CUST_SMOKE_001_ACC_1",
            customer_id="CUST_SMOKE_001",
            account_type="Checking",
            opening_date="2020-01-15",
            current_balance=1500.00,
            average_monthly_balance=1200.00,
            status="Active"
        )

        transactions = [
            TransactionData(
                transaction_id=f"TXN_SMOKE_00{i}",
                account_id="CUST_SMOKE_001_ACC_1",
                transaction_date=f"2024-12-{i:02d}",
                transaction_type="Cash_Deposit",
                amount=9800.00 - (i * 100),
                description="Cash deposit at branch",
                method="Teller",
                counterparty=None,
                location="Newark Branch"
            )
            for i in range(1, 4)
        ]

        case = CaseData(
            customer=customer,
            accounts=[account],
            transactions=transactions,
            data_sources={
                "customer_source":    "smoke_test",
                "account_source":     "smoke_test",
                "transaction_source": "smoke_test"
            }
        )
        print("✅ Step 2 — Test case built")

    except Exception as e:
        print(f"❌ Step 2 FAILED — Could not build test case: {e}")
        return

    # ── Step 3: Call analyze_case ─────────────────────────────────────────────
    try:
        result = agent.analyze_case(case)
        print("✅ Step 3 — analyze_case() returned without error")

    except Exception as e:
        print(f"❌ Step 3 FAILED — analyze_case() raised: {e}")
        return

    # ── Step 4: Validate output structure ─────────────────────────────────────
    failures = []

    if result.classification not in {"Structuring", "Sanctions", "Fraud", "Money_Laundering", "Other"}:
        failures.append(f"classification '{result.classification}' not in valid set")

    if not (0.0 <= result.confidence_score <= 1.0):
        failures.append(f"confidence_score {result.confidence_score} out of range")

    if result.risk_level not in {"Low", "Medium", "High", "Critical"}:
        failures.append(f"risk_level '{result.risk_level}' not in valid set")

    if not result.reasoning or not result.reasoning.strip():
        failures.append("reasoning is empty")

    if not result.key_indicators:
        failures.append("key_indicators list is empty")

    if failures:
        print("❌ Step 4 FAILED — Structural validation errors:")
        for f in failures:
            print(f"     • {f}")
        return

    print("✅ Step 4 — Output structure valid")

    # ── Step 5: Print summary ─────────────────────────────────────────────────
    print()
    print("📊 Result Summary:")
    print(f"   Classification : {result.classification}")
    print(f"   Risk Level     : {result.risk_level}")
    print(f"   Confidence     : {result.confidence_score:.2f}")
    print(f"   Indicators     : {len(result.key_indicators)} found")
    for indicator in result.key_indicators:
        print(f"     • {indicator}")
    print(f"   Reasoning      : {result.reasoning[:120]}{'...' if len(result.reasoning) > 120 else ''}")
    print()
    print(f"📋 Audit log entries: {len(logger.entries)}")
    print()
    print("✅ SMOKE TEST PASSED")
    print("=" * 45)


# Call it — client already exists from the previous cell
simple_risk_analyst_smoke_test(client)

🔍 Risk Analyst Smoke Test
✅ Step 1 — Agent initialized
✅ Step 2 — Test case built
✅ Step 3 — analyze_case() returned without error
✅ Step 4 — Output structure valid

📊 Result Summary:
   Classification : Structuring
   Risk Level     : High
   Confidence     : 0.95
   Indicators     : 3 found
     • Consecutive cash deposits below $10,000
     • High risk rating
     • Cashier occupation with low annual income
   Reasoning      : Customer with high risk rating made consecutive cash deposits just below $10,000 over three days, indicating potential s...

📋 Audit log entries: 1

✅ SMOKE TEST PASSED


### 🧪 Risk Analyst Testing Framework

In [7]:
# COMPREHENSIVE Risk Analyst Testing - Import Pre-Built Test Suite
# Use the project test suite to validate the agent

import sys
import os

# Add project root, src, and tests so test modules resolve the same code as notebook
project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
tests_path = os.path.join(project_root, 'tests')
for p in (project_root, src_path, tests_path):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 Python path: project_root={project_root}, src, tests")

def run_comprehensive_risk_analyst_tests():
    """
    Use the project test suite to validate the Risk Analyst Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Case analysis with valid inputs
    - JSON parsing and error handling
    - System prompt structure and content
    - API call parameters and responses
    - Helper method functionality
    """
    print("🧪 Comprehensive Risk Analyst Testing")
    print("✅ Risk Analyst Agent test (run when agent is implemented)")
    
    # Uncomment when the agent is ready:
    try:
        from test_risk_analyst import TestRiskAnalystAgent
        import pytest
        
        print("🔍 Loading comprehensive test suite...")
        
        # Run the test suite
        print("� Running Risk Analyst test suite...")
        _cwd = os.getcwd()
        try:
            os.chdir(project_root)
            result = pytest.main(["tests/test_risk_analyst.py", "-v", "--tb=short"])
        finally:
            os.chdir(_cwd)
        
        if result == 0:
            print("✅ All Risk Analyst tests passed!")
        else:
            print("❌ Some tests failed. Check the output above for details.")
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure you've implemented RiskAnalystAgent in src/risk_analyst_agent.py")

# Quick preview of available tests
try:
    from test_risk_analyst import TestRiskAnalystAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestRiskAnalystAgent) 
                   if method.startswith('test_')]
    
    print("📊 Preview of Comprehensive Risk Analyst Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestRiskAnalystAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate edge cases you might not think of!")
    print("💡 Much more thorough than manual testing!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_risk_analyst_tests()

📁 Python path: project_root=/Users/ak/Documents/aiforfin/project/starter, src, tests
📊 Preview of Comprehensive Risk Analyst Tests:
   • Test RiskAnalystAgent initializes properly
   • Test handling of invalid JSON response
   • Test successful case analysis with valid response
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   ... and 5 more tests

💡 These tests validate edge cases you might not think of!
💡 Much more thorough than manual testing!
🧪 Comprehensive Risk Analyst Testing
✅ Risk Analyst Agent test (run when agent is implemented)
🔍 Loading comprehensive test suite...
� Running Risk Analyst test suite...
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/ak/Documents/aiforfin/project/starter
plugins: anyio-4.10.0
collecting ... collected 10 items

tests/test_risk_analyst.py::T

## ✅ Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [8]:
def design_react_prompt() -> str:
    """Design and test ReACT prompt for compliance narratives."""

    system_prompt = """
You are a Senior Compliance Officer with 20+ years of experience in BSA/AML regulatory
reporting. You specialize in drafting Suspicious Activity Reports that meet FinCEN
requirements and withstand regulatory scrutiny.

You follow the ReACT framework for every case — explicit Reasoning before Action:

═══════════════════════════════════════════
REASONING PHASE — Think before you write
═══════════════════════════════════════════

REASON 1 - REVIEW RISK FINDINGS:
- What classification did the Risk Analyst assign?
- What is the confidence score and risk level?
- Which key indicators were flagged?
- Does the evidence support the classification?

REASON 2 - ASSESS REGULATORY REQUIREMENTS:
- Which regulations apply to this classification?
- BSA 31 CFR 1020.320: SAR mandatory filing triggers
- 12 CFR 21.11: National bank SAR obligations
- FinCEN SAR Instructions: Narrative completeness standards
- OFAC: Sanctions screening requirements if applicable

REASON 3 - IDENTIFY REQUIRED NARRATIVE ELEMENTS:
- WHO: Subject name, DOB, address, occupation
- WHAT: Specific suspicious activity and amounts
- WHEN: Date range of suspicious transactions
- WHERE: Account numbers, branch locations
- WHY: Why activity is suspicious relative to profile
- HOW: Transaction methods and patterns used

REASON 4 - PLAN NARRATIVE STRUCTURE:
- Open with the most suspicious activity
- Support with specific transaction details
- Connect to customer profile anomalies
- Close with filing basis and regulatory hook

═══════════════════════════════════════════
ACTION PHASE — Execute with precision
═══════════════════════════════════════════

ACTION 1 - DRAFT NARRATIVE:
- Write ≤120 words (regulatory standard for concise SARs)
- Use past tense, third person, active voice
- Include specific dollar amounts and dates
- Reference account IDs where relevant
- Avoid conclusions about guilt — report facts only

ACTION 2 - CITE REGULATIONS:
- List every applicable regulation by exact citation
- Match citations to the specific classification type
- Always include the base BSA SAR filing requirement

ACTION 3 - VERIFY COMPLETENESS:
- Confirm all 5 W's are addressed (Who/What/When/Where/Why)
- Confirm at least one regulation is cited
- Confirm narrative is factual and not speculative
- Set completeness_check = true only if ALL criteria met

═══════════════════════════════════════════
OUTPUT FORMAT — Strict JSON only
═══════════════════════════════════════════

CRITICAL: Respond with ONLY a valid JSON object. No explanation, no preamble,
no markdown. Just the raw JSON:

{
    "narrative": "Factual SAR narrative ≤120 words in regulatory language",
    "narrative_reasoning": "Your ReACT reasoning summary (max 500 chars)",
    "regulatory_citations": [
        "31 CFR 1020.320 (BSA SAR Filing)",
        "12 CFR 21.11 (National Bank SAR)",
        "FinCEN SAR Instructions"
    ],
    "completeness_check": true
}
"""
    return system_prompt


# ── Test and validate the prompt ─────────────────────────────────────────────
react_prompt = design_react_prompt()

required_elements = {
    "REASONING PHASE":       "ReACT reasoning block present",
    "ACTION PHASE":          "ReACT action block present",
    "WHO":                   "5 W's coverage — Who",
    "WHAT":                  "5 W's coverage — What",
    "WHEN":                  "5 W's coverage — When",
    "WHERE":                 "5 W's coverage — Where",
    "WHY":                   "5 W's coverage — Why",
    "≤120 words":            "Narrative word limit enforced",
    "narrative":             "JSON field: narrative",
    "narrative_reasoning":   "JSON field: narrative_reasoning",
    "regulatory_citations":  "JSON field: regulatory_citations",
    "completeness_check":    "JSON field: completeness_check",
    "31 CFR 1020.320":       "Base BSA citation present",
    "12 CFR 21.11":          "National bank SAR citation present",
    "FinCEN SAR":            "FinCEN instructions referenced",
}

print("⚡ ReACT Prompt Validation:")
print("=" * 50)

all_present = True
for element, description in required_elements.items():
    present = element in react_prompt
    status  = "✅" if present else "❌"
    print(f"  {status} {description}")
    if not present:
        all_present = False

print()
print(f"📏 Prompt length : {len(react_prompt):,} chars | {len(react_prompt.split())} words")
print()

if all_present:
    print("✅ ReACT prompt is complete — ready for compliance_officer_agent.py")
else:
    print("❌ Prompt missing required elements — review ❌ items above")

print()
print("✅ Prompt preview:")
print("-" * 50)
print(react_prompt[:300] + "...")

⚡ ReACT Prompt Validation:
  ✅ ReACT reasoning block present
  ✅ ReACT action block present
  ✅ 5 W's coverage — Who
  ✅ 5 W's coverage — What
  ✅ 5 W's coverage — When
  ✅ 5 W's coverage — Where
  ✅ 5 W's coverage — Why
  ✅ Narrative word limit enforced
  ✅ JSON field: narrative
  ✅ JSON field: narrative_reasoning
  ✅ JSON field: regulatory_citations
  ✅ JSON field: completeness_check
  ✅ Base BSA citation present
  ✅ National bank SAR citation present
  ✅ FinCEN instructions referenced

📏 Prompt length : 2,906 chars | 401 words

✅ ReACT prompt is complete — ready for compliance_officer_agent.py

✅ Prompt preview:
--------------------------------------------------

You are a Senior Compliance Officer with 20+ years of experience in BSA/AML regulatory
reporting. You specialize in drafting Suspicious Activity Reports that meet FinCEN
requirements and withstand regulatory scrutiny.

You follow the ReACT framework for every case — explicit Reasoning before Action...


In [9]:
from compliance_officer_agent import ComplianceOfficerAgent
from foundation_sar import (
    CustomerData, AccountData, TransactionData,
    CaseData, ExplainabilityLogger, RiskAnalystOutput
)

def simple_compliance_officer_smoke_test(client):
    print("✅ Compliance Officer Smoke Test")
    print("=" * 45)

    # Step 1: Initialize agent
    try:
        logger = ExplainabilityLogger(log_file="compliance_smoke_audit.jsonl")
        agent  = ComplianceOfficerAgent(
            openai_client=client,
            explainability_logger=logger,
            model="gpt-4o"
        )
        print("✅ Step 1 — Agent initialized")
    except Exception as e:
        print(f"❌ Step 1 FAILED: {e}")
        return

    # Step 2: Build test data
    try:
        customer = CustomerData(
            customer_id="CUST_SMOKE_CO_001",
            name="Test Subject",
            date_of_birth="1978-09-12",
            ssn_last_4="5678",
            address="123 Main St, Newark, NJ 07102",
            customer_since="2019-05-01",
            risk_rating="High",
            occupation="Cashier",
            annual_income=30000
        )
        account = AccountData(
            account_id="CUST_SMOKE_CO_001_ACC_1",
            customer_id="CUST_SMOKE_CO_001",
            account_type="Checking",
            opening_date="2019-05-15",
            current_balance=2000.00,
            average_monthly_balance=1800.00,
            status="Active"
        )
        transactions = [
            TransactionData(
                transaction_id=f"TXN_SMOKE_CO_{i}",
                account_id="CUST_SMOKE_CO_001_ACC_1",
                transaction_date=f"2024-12-{i:02d}",
                transaction_type="Cash_Deposit",
                amount=9700.00 - (i * 100),
                description="Cash deposit at branch",
                method="Teller",
                location="Newark Branch"
            )
            for i in range(1, 4)
        ]
        case = CaseData(
            customer=customer,
            accounts=[account],
            transactions=transactions,
            data_sources={
                "customer_source":    "smoke_test",
                "account_source":     "smoke_test",
                "transaction_source": "smoke_test"
            }
        )
        risk_analysis = RiskAnalystOutput(
            classification="Structuring",
            confidence_score=0.92,
            reasoning="Three cash deposits just below $10k threshold on consecutive days.",
            key_indicators=[
                "Cash deposits below $10,000 CTR threshold",
                "Consecutive daily pattern",
                "Income inconsistent with deposit volume"
            ],
            risk_level="High"
        )
        print("✅ Step 2 — Test data built")
    except Exception as e:
        print(f"❌ Step 2 FAILED: {e}")
        return

    # Step 3: Call generate_compliance_narrative
    try:
        result = agent.generate_compliance_narrative(case, risk_analysis)
        print("✅ Step 3 — generate_compliance_narrative() returned without error")
    except Exception as e:
        print(f"❌ Step 3 FAILED: {e}")
        return

    # Step 4: Validate output
    failures  = []
    word_count = len(result.narrative.split())

    if not result.narrative or not result.narrative.strip():
        failures.append("narrative is empty")
    if word_count > 120:
        failures.append(f"narrative exceeds 120 words ({word_count})")
    if not result.regulatory_citations:
        failures.append("no regulatory citations returned")
    if not result.narrative_reasoning or not result.narrative_reasoning.strip():
        failures.append("narrative_reasoning is empty")
    if not isinstance(result.completeness_check, bool):
        failures.append("completeness_check is not a boolean")

    if failures:
        print("❌ Step 4 FAILED — Validation errors:")
        for f in failures:
            print(f"     • {f}")
        return

    print("✅ Step 4 — Output structure valid")

    # Step 5: Print summary
    print()
    print("📊 Result Summary:")
    print(f"   Word Count   : {word_count}/120")
    print(f"   Completeness : {result.completeness_check}")
    print(f"   Citations    : {len(result.regulatory_citations)}")
    for cite in result.regulatory_citations:
        print(f"     • {cite}")
    print(f"   Preview      : {result.narrative[:100]}...")
    print()
    print(f"📋 Audit log entries: {len(logger.entries)}")
    print()
    print("✅ SMOKE TEST PASSED")
    print("=" * 45)


# client already initialized in a previous cell
simple_compliance_officer_smoke_test(client)

✅ Compliance Officer Smoke Test
✅ Step 1 — Agent initialized
✅ Step 2 — Test data built
✅ Step 3 — generate_compliance_narrative() returned without error
✅ Step 4 — Output structure valid

📊 Result Summary:
   Word Count   : 73/120
   Completeness : True
   Citations    : 3
     • 31 CFR 1020.320 (BSA SAR Filing)
     • 12 CFR 21.11 (National Bank SAR)
     • FinCEN SAR Instructions
   Preview      : Test Subject, DOB 1978-09-12, residing at 123 Main St, Newark, NJ, employed as a cashier, conducted ...

📋 Audit log entries: 1

✅ SMOKE TEST PASSED


### 🧪 Compliance Officer Testing Framework

In [10]:
# COMPREHENSIVE Compliance Officer Testing - Import Pre-Built Test Suite
# Use the project test suite to validate the agent

import sys
import os

# Add project root, src, and tests (same as Risk Analyst cell)
project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
tests_path = os.path.join(project_root, 'tests')
for p in (project_root, src_path, tests_path):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 Python path: project_root, src, tests")

def run_comprehensive_compliance_officer_tests():
    """
    Use the project test suite to validate the Compliance Officer Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Narrative generation with valid inputs
    - Word count limits (≤120 words)
    - Regulatory citations inclusion
    - JSON parsing and error handling
    - ReACT prompt structure validation
    """
    print(" Comprehensive Compliance Officer Testing")
    print("✅ Compliance Officer Agent test (run when agent is implemented)")
    
    # Uncomment when the agent is ready:
    try:
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print(" Loading comprehensive test suite...")
        
        # Run from project root so imports match
        print("🚀 Running Compliance Officer test suite...")
        _cwd = os.getcwd()
        try:
            os.chdir(project_root)
            result = pytest.main(["tests/test_compliance_officer.py", "-v", "--tb=short"])
        finally:
            os.chdir(_cwd)
        
        if result == 0:
            print("✅ All Compliance Officer tests passed!")
        else:
            print("❌ Some tests failed. Check the output above for details.")
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure you've implemented ComplianceOfficerAgent in src/compliance_officer_agent.py")

# Quick preview of available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestComplianceOfficerAgent) 
                   if method.startswith('test_')]
    
    print(" Preview of Comprehensive Compliance Officer Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate regulatory compliance requirements!")
    print("💡 Includes word limits, citations, and required elements!")
except Exception as e:
    print(f" Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_compliance_officer_tests()

📁 Python path: project_root, src, tests
 Preview of Comprehensive Compliance Officer Tests:
   • Test ComplianceOfficerAgent initializes properly
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   ... and 5 more tests

💡 These tests validate regulatory compliance requirements!
💡 Includes word limits, citations, and required elements!
 Comprehensive Compliance Officer Testing
✅ Compliance Officer Agent test (run when agent is implemented)
 Loading comprehensive test suite...
🚀 Running Compliance Officer test suite...
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/ak/Documents/aiforfin/project/starter
plugins: anyio-4.10.0
collecting ... collected 10 items

tests/test_compliance_officer.py::T

In [11]:
# COMPLETE AGENT TESTING - Two-Tier Approach
# Test both agents together

def complete_agent_testing_workflow():
    """
    Complete testing workflow using two-tier approach:
    
    TIER 1: Simple Smoke Tests (You write these)
    - Basic functionality verification
    - Quick sanity checks
    - Development debugging
    
    TIER 2: Comprehensive Test Suites (Pre-built for you)
    - Complex edge cases
    - Regulatory compliance validation
    - Professional-grade testing
    """
    print("🔬 Complete Agent Testing Workflow")
    print("=" * 50)
    
    print("\n✅ TIER 1: Simple Smoke Tests (DO FIRST)")
    print("   1. Write simple_risk_analyst_smoke_test() - verify basic functionality")
    print("   2. Write simple_compliance_officer_smoke_test() - verify basic functionality")
    print("   3. Fix any basic issues before moving to Tier 2")
    
    print("\n🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)")
    print("   1. Run comprehensive risk analyst test suite (10 comprehensive tests)")
    print("   2. Run comprehensive compliance officer test suite (10 comprehensive tests)")
    print("   3. Get detailed pass/fail results with specific feedback")
    
    print("\n💡 WHY THIS APPROACH?")
    print("   ✅ Tier 1: Quick feedback while developing")
    print("   ✅ Tier 2: Professional validation without writing complex tests")
    print("   ✅ Saves time: You focus on implementation, not test creation")
    print("   ✅ Better coverage: Our test suites test edge cases you might miss")

# Quick test runner when both agents are ready
def run_both_agents_quick_test():
    """Quick test of both agents using pre-built test suites"""
    import os
    import sys
    project_root = os.path.abspath('..')
    for p in (project_root, os.path.join(project_root, 'src'), os.path.join(project_root, 'tests')):
        if p not in sys.path:
            sys.path.insert(0, p)
    print("🚀 Quick Test of Both Agents")
    print("✅ Run pytest from project root when both agents are implemented")
    
    try:
        import pytest
        print("🔍 Running quick tests for both agents...")
        _cwd = os.getcwd()
        try:
            os.chdir(project_root)
            risk_result = pytest.main([
                "tests/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization",
                "tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success",
                "-v"
            ])
            compliance_result = pytest.main([
                "tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization", 
                "tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
                "-v"
            ])
        finally:
            os.chdir(_cwd)
        
        if risk_result == 0 and compliance_result == 0:
            print("🎉 Both agents working! Ready for full test suite testing!")
        else:
            print("⚠️ Fix issues before running comprehensive tests")
            if risk_result != 0:
                print("   🔍 Risk Analyst needs fixes")
            if compliance_result != 0:
                print("   📝 Compliance Officer needs fixes")
                
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure both agents are implemented")

complete_agent_testing_workflow()
run_both_agents_quick_test()

🔬 Complete Agent Testing Workflow

✅ TIER 1: Simple Smoke Tests (DO FIRST)
   1. Write simple_risk_analyst_smoke_test() - verify basic functionality
   2. Write simple_compliance_officer_smoke_test() - verify basic functionality
   3. Fix any basic issues before moving to Tier 2

🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)
   1. Run comprehensive risk analyst test suite (10 comprehensive tests)
   2. Run comprehensive compliance officer test suite (10 comprehensive tests)
   3. Get detailed pass/fail results with specific feedback

💡 WHY THIS APPROACH?
   ✅ Tier 1: Quick feedback while developing
   ✅ Tier 2: Professional validation without writing complex tests
   ✅ Saves time: You focus on implementation, not test creation
   ✅ Better coverage: Our test suites test edge cases you might miss
🚀 Quick Test of Both Agents
✅ Run pytest from project root when both agents are implemented
🔍 Running quick tests for both agents...
============================= test session sta

## 🔗 Next: Workflow Integration

Once both agents are working, integrate them in `03_workflow_integration.ipynb` for the full pipeline.

In [12]:
# Preview of integrated workflow (full flow in 03_workflow_integration.ipynb)
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent
- [ ] Implement Chain-of-Thought system prompt
- [ ] Create `analyze_case` method with error handling
- [ ] Add JSON parsing and validation
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Compliance Officer Agent
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify the agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**